In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

Salting with Join

In [0]:
skewed_transactions = [
    ("GUEST_01", "Laptop", 1200),
    ("GUEST_01", "Mouse", 25),
    ("GUEST_01", "Keyboard", 75),
    ("GUEST_01", "Monitor", 300),
    ("GUEST_01", "HDMI Cable", 15),
    ("GUEST_01", "Desk Mat", 20),
    ("USER_999", "Phone Case", 45),
    ("USER_888", "Charger", 30)
]
df_skewed = spark.createDataFrame(skewed_transactions, ["user_id", "item", "price"])

# The smaller lookup dataset 
user_profiles = [
    ("GUEST_01", "Anonymous Guest", "United States"),
    ("USER_999", "Alice Smith", "Canada"),
    ("USER_888", "Bob Jones", "United Kingdom")
]
df_user = spark.createDataFrame(user_profiles, ["user_id", "name", "country"])

print("--- Original Skewed Transactions ---")
df_skewed.show()

print("--- Original User Lookup Profiles ---")
df_user.show()


--- Original Skewed Transactions ---
+--------+----------+-----+
| user_id|      item|price|
+--------+----------+-----+
|GUEST_01|    Laptop| 1200|
|GUEST_01|     Mouse|   25|
|GUEST_01|  Keyboard|   75|
|GUEST_01|   Monitor|  300|
|GUEST_01|HDMI Cable|   15|
|GUEST_01|  Desk Mat|   20|
|USER_999|Phone Case|   45|
|USER_888|   Charger|   30|
+--------+----------+-----+

--- Original User Lookup Profiles ---
+--------+---------------+--------------+
| user_id|           name|       country|
+--------+---------------+--------------+
|GUEST_01|Anonymous Guest| United States|
|USER_999|    Alice Smith|        Canada|
|USER_888|      Bob Jones|United Kingdom|
+--------+---------------+--------------+



Applying Salting to the Skewed Data

In [0]:
# We will split the skewed transactions into 2 buckets( 0 & 1)
num_salts=2

df_skewed_salted=df_skewed.withColumn("salt",floor(rand()*num_salts))
# display(df_skewed_salted)

# Generating new composite key
df_skewed_salted=df_skewed_salted.withColumn("salted_user_id",concat(col("user_id"),lit("_"),col("salt")))

display(df_skewed_salted)




user_id,item,price,salt,salted_user_id
GUEST_01,Laptop,1200,1,GUEST_01_1
GUEST_01,Mouse,25,1,GUEST_01_1
GUEST_01,Keyboard,75,0,GUEST_01_0
GUEST_01,Monitor,300,0,GUEST_01_0
GUEST_01,HDMI Cable,15,1,GUEST_01_1
GUEST_01,Desk Mat,20,1,GUEST_01_1
USER_999,Phone Case,45,0,USER_999_0
USER_888,Charger,30,0,USER_888_0


Step 3: Replicate the lookup dataset

In [0]:
##create an array/range of our salt values [0,1]

from pyspark.sql.functions import lit
df_salts=spark.range(num_salts).withColumnRenamed("id", "salt")

# display(df_salts)

## Replicate every loopup row by cross joining it with my salt range

df_user_replicated=df_user.crossJoin(df_salts)
# display(df_user_replicated)

#Generate matching composite keys on the lookup side

df_user_replicated=df_user_replicated.withColumn("salted_user_id",concat(col("user_id"),lit("_"),col("salt")))
display(df_user_replicated)


user_id,name,country,salt,salted_user_id
GUEST_01,Anonymous Guest,United States,0,GUEST_01_0
GUEST_01,Anonymous Guest,United States,1,GUEST_01_1
USER_999,Alice Smith,Canada,0,USER_999_0
USER_999,Alice Smith,Canada,1,USER_999_1
USER_888,Bob Jones,United Kingdom,0,USER_888_0
USER_888,Bob Jones,United Kingdom,1,USER_888_1


##Joining on the balanced keys 

In [0]:
df_joined_salted=df_skewed_salted.join(df_user_replicated,on="salted_user_id",how="inner")
display(df_joined_salted)

salted_user_id,user_id,item,price,salt,user_id,name,country,salt
GUEST_01_1,GUEST_01,Laptop,1200,1,GUEST_01,Anonymous Guest,United States,1
GUEST_01_1,GUEST_01,Mouse,25,1,GUEST_01,Anonymous Guest,United States,1
GUEST_01_0,GUEST_01,Keyboard,75,0,GUEST_01,Anonymous Guest,United States,0
GUEST_01_0,GUEST_01,Monitor,300,0,GUEST_01,Anonymous Guest,United States,0
GUEST_01_1,GUEST_01,HDMI Cable,15,1,GUEST_01,Anonymous Guest,United States,1
GUEST_01_1,GUEST_01,Desk Mat,20,1,GUEST_01,Anonymous Guest,United States,1
USER_999_0,USER_999,Phone Case,45,0,USER_999,Alice Smith,Canada,0
USER_888_0,USER_888,Charger,30,0,USER_888,Bob Jones,United Kingdom,0
